[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/06_lora_qlora_math.ipynb)

# The math behind LoRA & QLoRA

> **Fine-tuning series — 06 of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. Conceptual/reference — no GPU setup needed.


## The math behind LoRA & QLoRA (deep dive)

### LoRA — low-rank weight updates

A pretrained linear layer applies $h = W_0 x$ with $W_0 \in \mathbb{R}^{d\times k}$.
Full fine-tuning learns an update $\Delta W$ so the layer becomes $h=(W_0+\Delta W)x$,
and $\Delta W$ has $d\cdot k$ free parameters — as big as $W_0$ itself.

**The bet (intrinsic rank).** LoRA assumes the *update* needed for adaptation has low
intrinsic rank, so it constrains

$$\Delta W = B A,\qquad B\in\mathbb{R}^{d\times r},\;\; A\in\mathbb{R}^{r\times k},\;\; r\ll\min(d,k).$$

Since $\operatorname{rank}(BA)\le r$, this is a rank-$r$ approximation of the full update.
The forward pass becomes

$$h = W_0 x + \frac{\alpha}{r}\,B A x,$$

where $\alpha$ is a fixed scale; the factor $\alpha/r$ keeps the update magnitude roughly
constant as you vary $r$, so the learning rate needn't be re-tuned.

**Initialization.** $A\sim\mathcal{N}(0,\sigma^2)$ and $B=0$, so at step 0 $BA=0$ and the
model is *exactly* the pretrained model — adaptation departs smoothly from the original.

**Gradients.** $W_0$ is frozen ($\partial L/\partial W_0$ is never applied). With
$s=\alpha/r$ and $g=\partial L/\partial h$,

$$\frac{\partial L}{\partial B}=s\,g\,(Ax)^\top,\qquad
  \frac{\partial L}{\partial A}=s\,B^\top g\,x^\top.$$

Only $A,B$ carry gradients and optimizer state — that is the memory saving.

**Parameter count.** Per matrix $\;d\cdot k \;\longrightarrow\; r\,(d+k)$. LoRA saves whenever
$r(d+k)<dk$, i.e. below the **break-even rank** $r<\dfrac{dk}{d+k}$. For a square
$4096\times4096$ projection that break-even is $2048$, so any practical $r$ (8/16/64) is a
massive reduction.

**Merging.** At inference fold it back, $W=W_0+\frac{\alpha}{r}BA$, computed once →
**zero added latency**; swapping $BA$ switches tasks on one frozen base.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

d, k, r = 4096, 4096, 8                 # a square attention projection, rank 8
A = rng.normal(size=(r, k)) * 0.01
B = np.zeros((d, r))                    # B = 0 at init -> dW = BA = 0
print("rank(dW) at init:", np.linalg.matrix_rank(B @ A), "(model == pretrained)")

B = rng.normal(size=(d, r)) * 0.01      # after training, B != 0
dW = B @ A
print("rank(dW) trained:", np.linalg.matrix_rank(dW), "(<= r =", r, ")")

full, lora = d * k, r * (d + k)
print(f"full dW params : {full:,}")
print(f"LoRA params    : {lora:,}  ->  {lora/full:.2%} of full ({full//lora}x fewer)")
print(f"break-even rank: r < dk/(d+k) = {d*k//(d+k)}")

### QLoRA — LoRA on a 4-bit base

QLoRA keeps LoRA's update but stores the frozen $W_0$ in 4-bit. Three ideas:

**(1) Block-wise quantization.** Split weights into blocks (size 64). For a block $w$ the
scale is the absmax $c=\max_i|w_i|$. Normalize $w/c\in[-1,1]$, snap each value to the
nearest of $2^4=16$ levels, and store a 4-bit index plus the scale:

$$\hat w_i = c\cdot Q\!\left(\frac{w_i}{c}\right),\qquad Q(\cdot)\in\{\text{16 levels}\}.$$

**(2) NF4 (NormalFloat4).** Pretrained weights are ~zero-mean Gaussian. Rather than evenly
spaced levels, NF4 places its 16 levels at the **quantiles of a standard normal**
(normalized to $[-1,1]$, symmetric, exact $0$) so each bin holds equal probability mass —
minimizing expected error for normal weights (the cell below shows NF4 beating linear int4).

**(3) Double quantization.** The per-block scales $c$ (one fp32 each) also cost memory, so DQ
quantizes *those* to 8-bit (with their own scale per 256 blocks):

$$\underbrace{4}_{\text{NF4 weight}}+\underbrace{\tfrac{32}{64}}_{\text{fp32 scale}}=4.5
\;\;\xrightarrow{\;\text{DQ}\;}\;\;
4+\tfrac{8}{64}+\tfrac{32}{64\cdot256}\approx 4.127\ \text{bits/param}.$$

**Computation — the actual trick.** Weights live in NF4; each tile is **dequantized to bf16
on the fly** for the matmul, and gradients flow **only into the bf16 adapters**:

$$h=\big(\operatorname{dequant}(W_0^{\text{NF4}})\big)x+\frac{\alpha}{r}BAx.$$

4-bit *storage*, 16-bit *compute*, adapters *trainable* — the frozen base is never
backpropagated into (you can't differentiate the quantizer anyway).

**Memory, 7B model.**

$$\text{bf16}=7\!\times\!10^9\cdot 2\,\text{B}=14\ \text{GB}
\;\;\to\;\;
\text{NF4}=7\!\times\!10^9\cdot 0.5\,\text{B}=3.5\ \text{GB}\;(\approx 3.6\ \text{GB with DQ}).$$

Plus a few MB of adapters and their optimizer state → a 7B QLoRA fine-tune fits on a single
consumer GPU, versus $\sim\!112$ GB for full mixed-precision Adam. **Paged optimizers** (a
systems trick, not math) page optimizer state to CPU on spikes to avoid OOM.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# The 16 NF4 levels = quantiles of a standard normal, normalized to [-1, 1].
nf4 = np.array([-1.0, -0.6961928, -0.5250731, -0.3949175, -0.2844414, -0.1847734,
                -0.0910500, 0.0, 0.0795803, 0.1609302, 0.2461123, 0.3379152,
                 0.4407098, 0.5626170, 0.7229568, 1.0], dtype=np.float32)

def quant_nf4(w):
    c = np.abs(w).max()                                    # block scale (absmax)
    idx = np.abs((w/c)[:, None] - nf4[None, :]).argmin(1)  # nearest level -> 4-bit index
    return idx.astype(np.uint8), c

def dequant_nf4(idx, c):
    return nf4[idx] * c

w = rng.normal(size=4096).astype(np.float32)               # ~Gaussian pretrained weights
idx, c = quant_nf4(w); w_hat = dequant_nf4(idx, c)

ci = np.abs(w).max(); w_lin = np.round(w/ci*7)/7*ci        # linear int4 baseline
print(f"NF4   mean|err| : {np.abs(w-w_hat).mean():.5f}")
print(f"int4  mean|err| : {np.abs(w-w_lin).mean():.5f}   (NF4 wins on normal data)")

P = 7e9
print(f"\n7B memory: bf16 {P*2/1e9:.0f} GB  ->  NF4 {P*0.5/1e9:.1f} GB  "
      f"(~{P*4.127/8/1e9:.1f} GB with double-quant)")